# 03. Experimentación y Selección de Modelos

Con los datos limpios, enriquecidos y escalados, es hora de encontrar el algoritmo predictivo ganador.

### Instrucciones Generales:
1. **Validación:** No entrenes y midas sobre el mismo conjunto (sobreajuste). Recuerda haber dividido en Entrenamiento y Prueba antes.
2. **Entrenamiento Base:** Entrena los siguientes modelos base con tu set de Entrenamiento y compáralos usando RMSE (Error Cuadrático Medio):
   - `LinearRegression`
   - `SGDRegressor`
   - `DecisionTreeRegressor`
   - `RandomForestRegressor`
3. **Cross Validation (Validación Cruzada):** Para tener una métrica robusta, usa `cross_val_score` en el set de Entrenamiento para cada uno de los modelos anteriores.
4. **Ajuste Fino (Fine Tuning):** Toma el modelo ganador y busca sus mejores hiperparámetros. Utiliza un `GridSearchCV` explorando el número de estimadores (`n_estimators`), las características máximas (`max_features`), etc.
5. **Conclusión y Benchmark (IMPORTANTE):** Redacta una conclusión comparando los algoritmos. Explica por qué escogiste el modelo final y valida tu decisión calculando el RMSE sobre tu Set de Prueba que habías reservado. Documenta si alguno de tus modelos se sobreajusto o subajusto. Recuerda que el modelo final no puede tener esos problemas!


### 1. Preparacion de Datos para Modelado

In [ ]:
# Empieza importando los algoritmos desde Scikit-Learn
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
import sys
sys.path.append("../src")
from features.build_features import preprocess_pipeline




In [8]:
# Cargamos el train y test 
train = pd.read_csv("../data/interim/train_set.csv")
test  = pd.read_csv("../data/interim/test_set.csv")

In [9]:
# Imprimismo las primeras filas para verificar que se cargaron correctamente
display(train.head())
display(test.head())

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.42,37.80,52.0,3321.0,1115.0,1576.0,1034.0,2.0987,458300.0,NEAR BAY
1,-118.38,34.14,40.0,1965.0,354.0,666.0,357.0,6.0876,483800.0,<1H OCEAN
2,-121.98,38.36,33.0,1083.0,217.0,562.0,203.0,2.4330,101700.0,INLAND
3,-117.11,33.75,17.0,4174.0,851.0,1845.0,780.0,2.2618,96100.0,INLAND
4,-118.15,33.77,36.0,4366.0,1211.0,1912.0,1172.0,3.5292,361800.0,NEAR OCEAN


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-121.95,37.11,21.0,2387.0,357.0,913.0,341.0,7.7360,397700.0,<1H OCEAN
1,-118.01,33.89,36.0,1589.0,265.0,804.0,272.0,4.6354,202900.0,<1H OCEAN
2,-118.18,33.74,30.0,5915.0,1750.0,2136.0,1503.0,4.0968,310000.0,NEAR OCEAN
3,-122.48,37.74,52.0,2166.0,423.0,1072.0,370.0,4.1310,314300.0,NEAR OCEAN
4,-122.39,37.78,5.0,1405.0,515.0,725.0,392.0,3.6037,187500.0,NEAR BAY


In [10]:
# Imprimimos la información general para verificar tipos de datos y valores nulos
display(train.info())
display(test.info())

<class 'pandas.DataFrame'>
RangeIndex: 16512 entries, 0 to 16511
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           16512 non-null  float64
 1   latitude            16512 non-null  float64
 2   housing_median_age  16512 non-null  float64
 3   total_rooms         16512 non-null  float64
 4   total_bedrooms      16344 non-null  float64
 5   population          16512 non-null  float64
 6   households          16512 non-null  float64
 7   median_income       16512 non-null  float64
 8   median_house_value  16512 non-null  float64
 9   ocean_proximity     16512 non-null  str    
dtypes: float64(9), str(1)
memory usage: 1.3 MB


None

<class 'pandas.DataFrame'>
RangeIndex: 4128 entries, 0 to 4127
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           4128 non-null   float64
 1   latitude            4128 non-null   float64
 2   housing_median_age  4128 non-null   float64
 3   total_rooms         4128 non-null   float64
 4   total_bedrooms      4089 non-null   float64
 5   population          4128 non-null   float64
 6   households          4128 non-null   float64
 7   median_income       4128 non-null   float64
 8   median_house_value  4128 non-null   float64
 9   ocean_proximity     4128 non-null   str    
dtypes: float64(9), str(1)
memory usage: 322.6 KB


None

In [11]:
# Ejecutamos el pipeline de limpieza y enriquecimiento en ambos datasets
train = preprocess_pipeline(train)
test  = preprocess_pipeline(test)

In [12]:
# Validamos que el pipeline se ejecutó correctamente y no se introdujeron valores nulos
display(train.info())
display(test.info())

<class 'pandas.DataFrame'>
RangeIndex: 16512 entries, 0 to 16511
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   longitude                   16512 non-null  float64
 1   latitude                    16512 non-null  float64
 2   housing_median_age          16512 non-null  float64
 3   total_rooms                 16512 non-null  float64
 4   total_bedrooms              16512 non-null  float64
 5   population                  16512 non-null  float64
 6   households                  16512 non-null  float64
 7   median_income               16512 non-null  float64
 8   median_house_value          16512 non-null  float64
 9   rooms_per_household         16512 non-null  float64
 10  bedrooms_per_room           16512 non-null  float64
 11  population_per_household    16512 non-null  float64
 12  dist_sf                     16512 non-null  float64
 13  dist_la                     16512 non-null

None

<class 'pandas.DataFrame'>
RangeIndex: 4128 entries, 0 to 4127
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   longitude                   4128 non-null   float64
 1   latitude                    4128 non-null   float64
 2   housing_median_age          4128 non-null   float64
 3   total_rooms                 4128 non-null   float64
 4   total_bedrooms              4128 non-null   float64
 5   population                  4128 non-null   float64
 6   households                  4128 non-null   float64
 7   median_income               4128 non-null   float64
 8   median_house_value          4128 non-null   float64
 9   rooms_per_household         4128 non-null   float64
 10  bedrooms_per_room           4128 non-null   float64
 11  population_per_household    4128 non-null   float64
 12  dist_sf                     4128 non-null   float64
 13  dist_la                     4128 non-null   

None

In [13]:
# Corregimos la posible desalineación de columnas entre train y test después del pipeline
train, test = train.align(test, join="left", axis=1, fill_value=0)

In [14]:
# Separamos las features de la variable objetivo del train
X_train = train.drop("median_house_value", axis=1)
y_train = train["median_house_value"]

In [15]:
# Separamos las features de la variable objetivo del test
X_test  = test.drop("median_house_value", axis=1)
y_test  = test["median_house_value"]

In [16]:
# Hacemos escalado de features para los modelos que lo requieran
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [17]:
# Imprimimos las primeras filas para verificar que se cargó correctamente
display(train.head())
display(test.head())

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,rooms_per_household,bedrooms_per_room,population_per_household,dist_sf,dist_la,income_per_room,ocean_proximity_<1H OCEAN,ocean_proximity_INLAND,ocean_proximity_ISLAND,ocean_proximity_NEAR BAY,ocean_proximity_NEAR OCEAN
0,-122.42,37.80,52.0,3321.0,1115.0,1576.0,1034.0,2.0987,458300.0,3.211799,0.335742,1.524178,0.030000,5.615594,0.653434,False,False,False,True,False
1,-118.38,34.14,40.0,1965.0,354.0,666.0,357.0,6.0876,483800.0,5.504202,0.180153,1.865546,5.431252,0.166433,1.105991,True,False,False,False,False
2,-121.98,38.36,33.0,1083.0,217.0,562.0,203.0,2.4330,101700.0,5.334975,0.200369,2.768473,0.736003,5.706461,0.456047,False,True,False,False,False
3,-117.11,33.75,17.0,4174.0,851.0,1845.0,780.0,2.2618,96100.0,5.351282,0.203881,2.365385,6.660068,1.169145,0.422665,False,True,False,False,False
4,-118.15,33.77,36.0,4366.0,1211.0,1912.0,1172.0,3.5292,361800.0,3.725256,0.277371,1.631399,5.850889,0.294109,0.947371,False,False,False,False,True


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,rooms_per_household,bedrooms_per_room,population_per_household,dist_sf,dist_la,income_per_room,ocean_proximity_<1H OCEAN,ocean_proximity_INLAND,ocean_proximity_ISLAND,ocean_proximity_NEAR BAY,ocean_proximity_NEAR OCEAN
0,-121.95,37.11,21.0,2387.0,357.0,913.0,341.0,7.7360,397700.0,7.000000,0.149560,2.677419,0.810247,4.809127,1.105143,True,False,False,False,False
1,-118.01,33.89,36.0,1589.0,265.0,804.0,272.0,4.6354,202900.0,5.841912,0.166772,2.955882,5.873883,0.280179,0.793473,True,False,False,False,False
2,-118.18,33.74,30.0,5915.0,1750.0,2136.0,1503.0,4.0968,310000.0,3.935462,0.295858,1.421158,5.849658,0.315753,1.040996,False,False,False,False,True
3,-122.48,37.74,52.0,2166.0,423.0,1072.0,370.0,4.1310,314300.0,5.854054,0.195291,2.897297,0.067082,5.620827,0.705665,False,False,False,False,True
4,-122.39,37.78,5.0,1405.0,515.0,725.0,392.0,3.6037,187500.0,3.584184,0.366548,1.849490,0.031623,5.579910,1.005445,False,False,False,True,False


### 2. Entrteanimento de Modelos Base

In [18]:
# Creamos una función para evaluar modelos con cross-validation
def evaluate_model(model, X, y):
    scores = cross_val_score(model, X, y, cv=5, scoring="neg_mean_squared_error")
    rmse_scores = np.sqrt(-scores)
    print(f"RMSE: {rmse_scores.mean():.4f} ± {rmse_scores.std():.4f}")
    return rmse_scores.mean()

In [19]:
# Definimos los modelos a evaluar
modelos={
    "Linear Regression": LinearRegression(),
    "SGDRegressor": SGDRegressor(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42)}

# Creamos un diccionario para almacenar los resultados
resultados = {}

In [20]:
# Evaluamos cada modelo y almacenamos los resultados
for nombre, modelo in modelos.items():
    print(f"Evaluando {nombre}")
    if nombre in ["Linear Regression", "SGDRegressor"]:
        rmse = evaluate_model(modelo, X_train_scaled, y_train)
    else:
        rmse = evaluate_model(modelo, X_train, y_train)
    resultados[nombre] = rmse

Evaluando Linear Regression...
RMSE: 67356.8256 ± 328.3337
Evaluando SGDRegressor...
RMSE: 227456575.5239 ± 149469218.3223
Evaluando Decision Tree...
RMSE: 70484.2747 ± 1067.7082
Evaluando Random Forest...
RMSE: 49382.2280 ± 582.5342


### 3. Fine Tuning del Mejor Modelo

In [ ]:
# Definimos el grid de hiperparámetros para Random Forest
param_grid = [
    {
        "n_estimators": [100, 200],
        "max_features": [6, 8, 10],
        "min_samples_leaf": [1, 3, 5],  
        "max_depth": [None, 20, 30],    
    },
]


In [28]:
# Ejecutamos GridSearchCV para encontrar los mejores hiperparámetros
grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    return_train_score=True
)

In [29]:
# Ajustamos el modelo con GridSearchCV
grid_search.fit(X_train, y_train)

print("Mejores parámetros:", grid_search.best_params_)
print("Mejor RMSE (CV):", f"${np.sqrt(-grid_search.best_score_):,.0f}")

Mejores parámetros: {'max_depth': 30, 'max_features': 6, 'min_samples_leaf': 1, 'n_estimators': 200}
Mejor RMSE (CV): $47,337


In [30]:
mejor_modelo = grid_search.best_estimator_

# Evaluamos contra el test set (solo una vez, al final)
y_pred = mejor_modelo.predict(X_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"RMSE en Test Set: ${rmse_test:,.0f}")

# Comparamos train vs test para detectar sobreajuste
y_pred_train = mejor_modelo.predict(X_train)
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
print(f"RMSE en Train Set: ${rmse_train:,.0f}")
print(f"Diferencia: ${rmse_test - rmse_train:,.0f}")


RMSE en Test Set: $47,416
RMSE en Train Set: $17,212
Diferencia: $30,204


### Benchmark y Conclusión Final

#### Comparación de modelos base (Cross-Validation, 5 folds)
| Modelo | RMSE Promedio | Desv. Estándar | Observación |
|---|---|---|---|
| LinearRegression | $67,357 | ± $328 | Estable pero limitado — la relación no es lineal |
| SGDRegressor | $227,456,576 | ± $149,469,218 | Falló completamente — muy sensible a la escala de los datos |
| DecisionTree | $70,484 | ± $1,068 | Peor que regresión lineal — memoriza pero no generaliza |
| **RandomForest** | **$49,382** | **± $583** | **Ganador — menor error y mayor estabilidad** |

#### Hallazgos importantes del entrenamiento base
- **SGDRegressor** colapsó con un RMSE de $227 millones a pesar del escalado,
  este modelo es extremadamente sensible a los hiperparámetros y requiere una
  configuración muy fina para funcionar; con parámetros por defecto no es viable
- **DecisionTree** tuvo peor desempeño que la regresión lineal, confirmando que
  un árbol solo sin restricciones memoriza el train sin aprender patrones reales
- **RandomForest** superó a todos por más de $17,000 de RMSE gracias a que
  combina cientos de árboles, reduciendo el error individual de cada uno

#### Fine-Tuning del modelo ganador — RandomForest
Se exploró una grilla de hiperparámetros con 5 folds de validación cruzada.
Los mejores parámetros encontrados fueron:
- `n_estimators: 200` — 200 árboles dan más estabilidad que 100
- `max_features: 6` — cada árbol ve solo 6 de las 19 variables, forzando diversidad
- `max_depth: 30` — profundidad controlada para evitar memorización extrema
- `min_samples_leaf: 1` — con `max_depth: 30` ya hay suficiente restricción

#### Benchmark final sobre datos nunca vistos
| Métrica | Valor |
|---|---|
| RMSE Cross-Validation | $47,337 |
| RMSE Test Set | $47,416 |
| RMSE Train Set | $17,212 |
| Diferencia CV vs Test | **$79** |

#### ¿Hay sobreajuste o subajuste?
- **No hay sobreajuste real**: la diferencia entre el RMSE de validación cruzada
  ($47,337) y el RMSE en test ($47,416) es de apenas $79 lo que significa que el modelo generaliza
  correctamente a datos nuevos
- El Train RMSE más bajo ($17,212) es comportamiento normal del RandomForest, ya que
  los árboles recuerdan parte del entrenamiento, pero lo que importa es que
  CV y Test sean consistentes, y en este caso son prácticamente iguales
- **SGDRegressor y DecisionTree** presentaron los mayores problemas:
  el primero por inestabilidad numérica y el segundo por sobreajuste

- Comentar que en un principio me dio un sobreajuste por lo que me toco agregarle mas restricciones a los árboles, pero con el ajuste fino logre controlar ese problema sin sacrificar la capacidad predictiva del modelo

#### Conclusión ejecutiva
El modelo **RandomForest con 200 árboles** es el elegido para producción.
Predice el precio de vivienda en California con un error promedio de **±$47,416**
sobre un precio medio de $206,333, representando un error relativo de **aproximadamente 23%**.
Este margen es aceptable dado que los datos tienen un techo artificial en $500,001
que impide al modelo aprender el comportamiento real del segmento de lujo.

